In [8]:
import torch
import os
import sys
import argparse
import numpy as np

helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/oldver/NRAD/non-resonant-AD/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
from SimpleMAF import SimpleMAF

In [4]:
seed = 2
data_path = f"SemiVisJets/data/data_seed{seed}"
mc_path = "SemiVisJets/data"
model_path = "SemiVisJets/models"
# config_path = "oldver/NRAD/non-resonant-AD/Train_Models/configs"
samples_path = "SemiVisJets/samples"

In [2]:
print("Setting up device...")
CUDA = torch.cuda.is_available()
print("cuda available:", CUDA)
device = torch.device("cuda" if CUDA else "cpu")

Setting up device...
cuda available: True


In [22]:
print("Loadding MC events...")
mc_events = np.load(f"{mc_path}/mc_events_chunk{seed:02d}.npz", allow_pickle=True)
mc_events_cr = mc_events["mc_events_cr"]
mc_events_sr = mc_events["mc_events_sr"]

print(mc_events_cr.shape, mc_events_sr.shape)

Loadding MC events...
(9952849, 7) (47151, 7)


In [13]:
n_context = 2
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")


    data_cr_test = data_events_cr[:, :n_context]
    mc_cr_test = mc_events_cr[:, :n_context]

    print("Loading model... at data chunk", i)
    model = "context_weight"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    print("Model path:", model_path_full)
    NN_Context_Weights = torch.load(model_path_full, weights_only=False)
    NN_Context_Weights.to(device)

    print("Generating samples... at data chunk", i)
    w_cr = NN_Context_Weights.evaluation(mc_cr_test)
    w_cr = (w_cr/(1-w_cr)).flatten()

    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz", target_cr = data_cr_test, mc_cr = mc_cr_test, w_cr = np.nan_to_num(w_cr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz")

    mc_events_sr = mc_events_sr[:, :n_context]
    w_sr = NN_Context_Weights.evaluation(mc_events_sr)
    w_sr = (w_sr/(1-w_sr)).flatten()
    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz", mc_samples = mc_events_sr, w_sr = np.nan_to_num(w_sr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz")

print("Done!")


Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Loading model... at data chunk 1
Model path: SemiVisJets/models/context_weight_MC02_Data01.pt
Generating samples... at data chunk 1
Saved samples to SemiVisJets/samples/context_weight_MC02_Data01_CR_samples.npz
Saved samples to SemiVisJets/samples/context_weight_MC02_Data01_SR_samples.npz
Loading data chunk 2
CR has 9983941 data events, 9952849 MC events.
Loading model... at data chunk 2
Model path: SemiVisJets/models/context_weight_MC02_Data02.pt
Generating samples... at data chunk 2
Saved samples to SemiVisJets/samples/context_weight_MC02_Data02_CR_samples.npz
Saved samples to SemiVisJets/samples/context_weight_MC02_Data02_SR_samples.npz
Loading data chunk 3
CR has 9983542 data events, 9952849 MC events.
Loading model... at data chunk 3
Model path: SemiVisJets/models/context_weight_MC02_Data03.pt
Generating samples... at data chunk 3
Saved samples to SemiVisJets/samples/context_weight_MC02_Data03_CR_samples.npz
Saved

In [23]:
# n_context = 2
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    data_events_sr = data_chunk["data_events_sr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")


    data_cr_test = data_events_cr
    data_sr_test = data_events_sr
    mc_cr_test = mc_events_cr

    print("Loading model... at data chunk", i)
    model = "reweight"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    print("Model path:", model_path_full)
    NN_Reweight = torch.load(model_path_full, weights_only=False)
    NN_Reweight.to(device)

    print("Generating samples... at data chunk", i)
    w_cr = NN_Reweight.evaluation(mc_cr_test)
    w_cr = (w_cr/(1-w_cr)).flatten()

    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz", target_cr = data_cr_test, mc_cr = mc_cr_test, w_cr = np.nan_to_num(w_cr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz")

    mc_events_sr = mc_events_sr
    w_sr = NN_Reweight.evaluation(mc_events_sr)
    w_sr = (w_sr/(1-w_sr)).flatten()
    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz", data_sr = data_sr_test, mc_samples = mc_events_sr, w_sr = np.nan_to_num(w_sr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz")

print("Done!")


Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Loading model... at data chunk 1
Model path: SemiVisJets/models/reweight_MC02_Data01.pt
Generating samples... at data chunk 1
Saved samples to SemiVisJets/samples/reweight_MC02_Data01_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Data01_SR_samples.npz
Loading data chunk 2
CR has 9983941 data events, 9952849 MC events.
Loading model... at data chunk 2
Model path: SemiVisJets/models/reweight_MC02_Data02.pt
Generating samples... at data chunk 2
Saved samples to SemiVisJets/samples/reweight_MC02_Data02_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Data02_SR_samples.npz
Loading data chunk 3
CR has 9983542 data events, 9952849 MC events.
Loading model... at data chunk 3
Model path: SemiVisJets/models/reweight_MC02_Data03.pt
Generating samples... at data chunk 3
Saved samples to SemiVisJets/samples/reweight_MC02_Data03_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Da

In [26]:
n_context = 2
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    data_events_sr = data_chunk["data_events_sr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")

    data_context_cr_test = data_events_cr[:, :n_context]
    data_feature_cr_test = data_events_cr[:, n_context:]
    data_feature_sr_test = data_events_sr[:, n_context:]
    mc_context_sr = mc_events_sr[:, :n_context]


    print("Loading model... at data chunk", i)
    model = "generate"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    print("Model path:", model_path_full)
    MAF = torch.load(model_path_full, weights_only=False)
    MAF.to(device)

    print("Generating samples... at data chunk", i)
    pred_bkg_CR = MAF.sample(1, data_context_cr_test)
    np.savez(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz", target_cr = data_feature_cr_test, generate_cr = pred_bkg_CR)
    oversample = 1
    pred_bkg_SR = MAF.sample(oversample, mc_context_sr)
    np.savez(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz", data_sr = data_feature_sr_test, samples = pred_bkg_SR)

    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz")
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz")

print("Done!")

Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Loading model... at data chunk 1
Model path: SemiVisJets/models/generate_MC02_Data01.pt
Generating samples... at data chunk 1


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.76 GiB. GPU 0 has a total capacity of 15.70 GiB of which 816.06 MiB is free. Process 52776 has 326.00 MiB memory in use. Including non-PyTorch memory, this process has 13.61 GiB memory in use. Of the allocated memory 10.50 GiB is allocated by PyTorch, and 2.83 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)